In [1]:
import pandas as pd

# Load your processed datasets
nasa_df = pd.read_csv("../Dataset/processed/nasa_processed.csv")
iot_df = pd.read_csv("../Dataset/processed/iot_daily.csv")

# Convert dates
nasa_df['date'] = pd.to_datetime(nasa_df['date'])
iot_df['date'] = pd.to_datetime(iot_df['date'])

In [2]:
locations = nasa_df['location'].unique()
print(locations)

['bangalore' 'chennai' 'delhi' 'kolkata']


In [3]:
iot_expanded = []

for loc in locations:
    temp = iot_df.copy()
    temp['location'] = loc
    iot_expanded.append(temp)

iot_df_expanded = pd.concat(iot_expanded).reset_index(drop=True)

print(iot_df_expanded['location'].unique())

['bangalore' 'chennai' 'delhi' 'kolkata']


In [4]:
nasa_df = nasa_df.sort_values(['date', 'location']).reset_index(drop=True)
iot_df_expanded = iot_df_expanded.sort_values(['date', 'location']).reset_index(drop=True)

In [5]:
nasa_df['date'] = pd.to_datetime(nasa_df['date'])
iot_df_expanded['date'] = pd.to_datetime(iot_df_expanded['date'])

In [6]:
nasa_df = nasa_df.dropna(subset=['date'])
iot_df_expanded = iot_df_expanded.dropna(subset=['date'])

In [6]:
nasa_df['date'] = pd.to_datetime(nasa_df['date'])
iot_df_expanded['date'] = pd.to_datetime(iot_df_expanded['date'])

In [7]:
df = pd.merge_asof(
    nasa_df,
    iot_df_expanded,
    on='date',
    by='location',
    direction='nearest',
    tolerance=pd.Timedelta('1D')
)

print("Shape:", df.shape)

Shape: (9132, 27)


In [8]:
df = df.sort_values(['location', 'date'])

df = df.groupby('location').apply(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
).reset_index(drop=True)

C:\Users\rajor\AppData\Local\Temp\ipykernel_10220\3775083366.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_10220\3775083366.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('location').apply(


In [9]:
df_agro = pd.read_csv("../Dataset/processed/agro_processed.csv")

In [10]:
agro_features = df_agro.drop(
    ['failure_flag', 'stress_level', 'suitability_score'],
    axis=1
)

In [11]:
agro_mean = agro_features.mean()

In [12]:
for col in agro_mean.index:
    df[col] = agro_mean[col]

In [13]:
df_smart = pd.read_csv("../Dataset/processed/smart_farming_processed.csv")

df_smart['mid_date'] = pd.to_datetime(df_smart['mid_date'])

In [14]:
df = df.sort_values(['date', 'location'])
df_smart = df_smart.sort_values('mid_date')

In [15]:
df = pd.merge_asof(
    df,
    df_smart,
    left_on='date',
    right_on='mid_date',
    direction='nearest',
    tolerance=pd.Timedelta('7D')  # crop cycles are longer
)

In [17]:
print(df.columns.tolist())

['date', 'location', 'temperature_x', 'humidity_x', 'solar_radiation', 'temp_change_x', 'temp_3d_avg_x', 'solar_temp_interaction', 'humidity_stress', 'temp_lag1', 'humidity_lag2', 'temperature_y', 'humidity_y', 'water_level', 'N', 'P', 'K', 'temp_humidity_index', 'water_stress', 'npk_sum', 'fan_active', 'irrigation_active', 'irrigation_events', 'temp_change_y', 'humidity_change', 'temp_3d_avg_y', 'water_7d_avg', 'soil_type', 'bulk_density', 'organic_matter_pct', 'cation_exchange_capacity', 'salinity_ec', 'buffering_capacity', 'soil_moisture_pct', 'moisture_limit_dry', 'moisture_limit_wet', 'moisture_regime', 'soil_temp_c', 'air_temp_c', 'thermal_regime', 'light_intensity_par', 'soil_ph', 'ph_stress_flag', 'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm', 'nutrient_balance', 'plant_category', 'fertility_index', 'nutrient_total', 'np_ratio', 'nk_ratio', 'pk_ratio', 'moisture_range', 'moisture_stress', 'temp_diff', 'thermal_stress_index', 'ph_deviation_x', 'salinity_stress', 'total_stres

In [18]:
# Climate (NASA)
df['temperature'] = df['temperature_x']
df['humidity'] = df['humidity_x']

# Soil moisture (Agro preferred)
df['soil_moisture'] = df['soil_moisture_pct']

# pH
df['soil_ph_clean'] = df['soil_ph']

# Keep rainfall as is
# rainfall_mm already exists

In [19]:
cols_to_drop = [
    'temperature_x', 'temperature_y', 'temperature_C',
    'humidity_x', 'humidity_y',
    'soil_moisture_pct',
    'ph_deviation_x', 'ph_deviation_y',
    'temp_change_x', 'temp_change_y',
    'temp_3d_avg_x', 'temp_3d_avg_y'
]

df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

In [20]:
print(df.columns)

Index(['date', 'location', 'solar_radiation', 'solar_temp_interaction',
       'humidity_stress', 'temp_lag1', 'humidity_lag2', 'water_level', 'N',
       'P', 'K', 'temp_humidity_index', 'water_stress', 'npk_sum',
       'fan_active', 'irrigation_active', 'irrigation_events',
       'humidity_change', 'water_7d_avg', 'soil_type', 'bulk_density',
       'organic_matter_pct', 'cation_exchange_capacity', 'salinity_ec',
       'buffering_capacity', 'moisture_limit_dry', 'moisture_limit_wet',
       'moisture_regime', 'soil_temp_c', 'air_temp_c', 'thermal_regime',
       'light_intensity_par', 'soil_ph', 'ph_stress_flag', 'nitrogen_ppm',
       'phosphorus_ppm', 'potassium_ppm', 'nutrient_balance', 'plant_category',
       'fertility_index', 'nutrient_total', 'np_ratio', 'nk_ratio', 'pk_ratio',
       'moisture_range', 'moisture_stress', 'temp_diff',
       'thermal_stress_index', 'salinity_stress', 'total_stress_index',
       'farm_id', 'region', 'crop_type', 'soil_moisture', 'soil_pH',


In [22]:
if 'soil_moisture_pct' in df.columns:
    df['soil_moisture_clean'] = df['soil_moisture_pct']
elif 'soil_moisture' in df.columns:
    df['soil_moisture_clean'] = df['soil_moisture']
else:
    raise ValueError("No soil moisture column found")

In [24]:
df['soil_moisture_clean']

0       5.018966e-17
1       5.018966e-17
2       5.018966e-17
3       5.018966e-17
4       5.018966e-17
            ...     
9127    5.018966e-17
9128    5.018966e-17
9129    5.018966e-17
9130    5.018966e-17
9131    5.018966e-17
Name: soil_moisture_clean, Length: 9132, dtype: float64

In [26]:
df['env_index'] = (
    df['temperature'] +
    df['humidity'] +
    df['soil_moisture_clean']
)

In [27]:
df['combined_stress'] = (
    df['temperature'] / (df['soil_moisture_clean'] + 1e-5)
)

In [41]:
df['fungal_risk'] = df['humidity'] * df['rainfall_mm']

In [42]:
df['ndvi_env'] = df['NDVI_index'] * df['humidity']

In [28]:
df['temp_x_moisture'] = (
    df['temperature'] * df['soil_moisture_clean']
)

In [43]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

In [44]:
df['temp_lag3'] = df.groupby('location')['temperature'].shift(3)
df['humidity_lag3'] = df.groupby('location')['humidity'].shift(3)
df['moisture_lag3'] = df.groupby('location')['soil_moisture_clean'].shift(3)

In [45]:
df = df.sort_values(['location', 'date'])

df = df.groupby('location').apply(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
).reset_index(drop=True)

C:\Users\rajor\AppData\Local\Temp\ipykernel_10220\3775083366.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_10220\3775083366.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('location').apply(


In [46]:
final_features = [
    # Core
    'temperature', 'humidity', 'solar_radiation',
    'rainfall_mm', 'soil_moisture_clean',

    # IoT
    'water_level', 'N', 'P', 'K',
    'npk_sum', 'temp_humidity_index', 'water_stress',
    'fan_active', 'irrigation_active', 'irrigation_events',

    # Soil
    'bulk_density', 'organic_matter_pct',
    'cation_exchange_capacity', 'salinity_ec',
    'buffering_capacity', 'soil_ph_clean',
    'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm',
    'fertility_index', 'nutrient_total',
    'np_ratio', 'nk_ratio', 'pk_ratio',
    'moisture_range', 'moisture_stress',
    'thermal_stress_index', 'salinity_stress',
    'total_stress_index',

    # Crop
    'NDVI_index', 'yield_per_day',
    'climate_score', 'water_efficiency',
    'ndvi_health_index', 'input_intensity',
    'heat_stress',

    # Engineered
    'env_index', 'combined_stress',
    'fungal_risk', 'ndvi_env', 'temp_x_moisture',

    # Time
    'day_of_week', 'month',

    # Lag
    'temp_lag3', 'humidity_lag3', 'moisture_lag3'
]

In [47]:
df['target'] = (
    (df['combined_stress'] > 30) |
    (df['fungal_risk'] > 5000)
).astype(int)

In [48]:
final_df = df[final_features + ['target']]

print("Final shape:", final_df.shape)
final_df.head()

Final shape: (9132, 52)


,temperature,humidity,solar_radiation,rainfall_mm,soil_moisture_clean,water_level,N,P,K,npk_sum,...,combined_stress,fungal_risk,ndvi_env,temp_x_moisture,day_of_week,month,temp_lag3,humidity_lag3,moisture_lag3,target
0,23.05,74.76,11.64,0.775321,5.018966e-17,100.0,185.0,190.0,160.0,535.0,...,2.305000e+06,42.12318,52.070753,1.156872e-15,2,1,23.05,74.76,5.018966e-17,1
1,23.29,73.83,14.50,0.775321,5.018966e-17,100.0,185.0,190.0,160.0,535.0,...,2.329000e+06,42.12318,52.070753,1.168917e-15,3,1,23.05,74.76,5.018966e-17,1
2,24.06,71.56,17.82,0.775321,5.018966e-17,100.0,185.0,190.0,160.0,535.0,...,2.406000e+06,42.12318,52.070753,1.207563e-15,4,1,23.05,74.76,5.018966e-17,1
3,24.46,71.99,17.08,0.775321,5.018966e-17,100.0,185.0,190.0,160.0,535.0,...,2.446000e+06,42.12318,52.070753,1.227639e-15,5,1,23.05,74.76,5.018966e-17,1
4,24.23,73.33,16.80,0.775321,5.018966e-17,100.0,185.0,190.0,160.0,535.0,...,2.423000e+06,42.12318,52.070753,1.216095e-15,6,1,23.29,73.83,5.018966e-17,1


In [49]:
final_df.to_csv("../Dataset/processed/final_dataset.csv", index=False)